# Exercise 3 — Intrinsic vs. Post-hoc Explainability

**PCS956 • Module C • Week 1**

## Research question

**How does direct inspection of an intrinsically interpretable model differ from explaining a more complex model with a post-hoc method?**

In this exercise, we will:

- use the same Titanic prediction task as in the previous exercises;
- fit a small decision tree and inspect it directly;
- fit a Random Forest as a less directly inspectable model;
- compare their predictive performance;
- use SHAP to explain the Random Forest after training;
- inspect one local SHAP explanation;
- check the SHAP additivity relationship; and
- compare intrinsic interpretation with post-hoc explanation.

The aim is to compare two different sources of explanation:

- **intrinsic interpretation** — understanding comes directly from the fitted model structure;
- **post-hoc explanation** — an additional method is applied after training to investigate model behaviour.


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier


## 2. Prepare the Titanic data

We use the same variables and preprocessing strategy as in Exercises 1 and 2, so the exploratory dataset visualisations are not repeated here.

The target is:

- `survived = 0` — did not survive;
- `survived = 1` — survived.

Categorical variables are one-hot encoded and all categories are retained.

Therefore:

- `sex` becomes `sex_female` and `sex_male`;
- `embarked` becomes `embarked_C`, `embarked_Q`, and `embarked_S`.

Because `sex_female` and `sex_male` are complementary encodings of the same original binary variable, a model may use either one to represent essentially the same partition.


In [ ]:
RANDOM_STATE = 42

# Load the Titanic dataset.
titanic = sns.load_dataset("titanic")

FEATURES = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

TARGET = "survived"

# Predictor matrix and target.
X = titanic[FEATURES].copy()
y = titanic[TARGET].copy()

# Split before fitting any preprocessing.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

numeric_features = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "sex",
    "embarked"
]

# Numerical preprocessing:
# replace missing values with the training-set median.
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical preprocessing:
# replace missing values with the most frequent category,
# then one-hot encode all observed categories.
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ],
    verbose_feature_names_out=False
)

# Fit preprocessing only on the training set.
X_train = preprocessor.fit_transform(X_train_raw)

# Apply the fitted preprocessing to the test set.
X_test = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train,
    columns=feature_names,
    index=X_train_raw.index
)

X_test_df = pd.DataFrame(
    X_test,
    columns=feature_names,
    index=X_test_raw.index
)

print("Training observations:", len(X_train_df))
print("Test observations:", len(X_test_df))
print("Transformed features:")
print(list(feature_names))


## 3. Model A — Small decision tree

We first fit a deliberately small decision tree.

Because the complete fitted structure can be inspected directly, this model provides an example of **intrinsic interpretability**.


In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE
)

tree_model.fit(
    X_train_df,
    y_train
)

tree_train_pred = tree_model.predict(
    X_train_df
)

tree_test_pred = tree_model.predict(
    X_test_df
)

print(
    "Tree training accuracy:",
    round(
        accuracy_score(
            y_train,
            tree_train_pred
        ),
        3
    )
)

print(
    "Tree test accuracy:",
    round(
        accuracy_score(
            y_test,
            tree_test_pred
        ),
        3
    )
)

print(
    "Tree test balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            tree_test_pred
        ),
        3
    )
)


In [ ]:
plt.figure(figsize=(20, 10))

plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[
        "did not survive",
        "survived"
    ],
    filled=True,
    rounded=True,
    impurity=True,
    precision=2
)

plt.title(
    "Small decision tree"
)

plt.show()


### Why this is intrinsic interpretation

The explanation comes from the fitted model itself.

We can inspect:

- the complete tree structure;
- the split features and thresholds;
- the leaves;
- and the exact path used for an individual prediction.

No additional explanation algorithm is required to inspect these aspects of the tree.


## 4. Model B — Random Forest

We now fit a Random Forest on the same transformed training data.

A Random Forest combines many decision trees. Although each tree can be inspected individually, the complete ensemble is much harder to understand directly as a single global structure.

This makes it useful for illustrating **post-hoc explainability**.


In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

forest_model.fit(
    X_train_df,
    y_train
)

forest_train_pred = forest_model.predict(
    X_train_df
)

forest_test_pred = forest_model.predict(
    X_test_df
)

print(
    "Random Forest training accuracy:",
    round(
        accuracy_score(
            y_train,
            forest_train_pred
        ),
        3
    )
)

print(
    "Random Forest test accuracy:",
    round(
        accuracy_score(
            y_test,
            forest_test_pred
        ),
        3
    )
)

print(
    "Random Forest test balanced accuracy:",
    round(
        balanced_accuracy_score(
            y_test,
            forest_test_pred
        ),
        3
    )
)


## 5. Compare predictive performance

The two models solve the same classification task, but they differ substantially in structural complexity.

The comparison below is descriptive for this train/test split. It should not be interpreted as evidence that one model family is universally more accurate than the other.


In [ ]:
performance_table = pd.DataFrame({
    "model": [
        "Decision Tree",
        "Random Forest"
    ],
    "training_accuracy": [
        accuracy_score(
            y_train,
            tree_train_pred
        ),
        accuracy_score(
            y_train,
            forest_train_pred
        )
    ],
    "test_accuracy": [
        accuracy_score(
            y_test,
            tree_test_pred
        ),
        accuracy_score(
            y_test,
            forest_test_pred
        )
    ],
    "test_balanced_accuracy": [
        balanced_accuracy_score(
            y_test,
            tree_test_pred
        ),
        balanced_accuracy_score(
            y_test,
            forest_test_pred
        )
    ]
})

display(
    performance_table
)


### Interpretation

Predictive performance tells us how the models behave on held-out data.

It does **not** tell us whether the internal decision process is easy to inspect.

The small tree and the Random Forest therefore differ along at least two dimensions:

- predictive behaviour;
- practical inspectability.


## 6. Post-hoc explanation of the Random Forest with SHAP

We now apply SHAP **after the Random Forest has been trained**.

For tree-based models, `TreeExplainer` is a specialised SHAP explainer that uses properties of tree ensembles.

The SHAP values describe how input features contribute to the explained model output relative to the explainer's reference value.

They should be interpreted as explanations of the fitted model, not as causal effects.


In [ ]:
# Create a SHAP explainer specialised for tree models.
explainer = shap.TreeExplainer(
    forest_model
)

# Explain the transformed test observations.
shap_values = explainer(
    X_test_df
)

print(
    "SHAP values shape:",
    shap_values.values.shape
)


### Which output are we explaining?

For a binary classifier, the explainer may return values for both classes.

In this exercise, we focus on the output associated with:

> **class 1 — survived**

The code below extracts the SHAP values for that class in a version-robust way.


In [ ]:
# Extract SHAP values for class 1 ("survived").
#
# Depending on the SHAP version, class outputs may be stored
# either in the final dimension of one Explanation object
# or as a list-like collection.

if shap_values.values.ndim == 3:
    # Shape: observations × features × classes
    survived_shap = shap.Explanation(
        values=shap_values.values[:, :, 1],
        base_values=shap_values.base_values[:, 1],
        data=shap_values.data,
        feature_names=shap_values.feature_names
    )
else:
    # Some configurations may already return a 2D Explanation.
    survived_shap = shap_values

print(
    "Class-1 SHAP matrix shape:",
    survived_shap.values.shape
)


## 7. Global SHAP summary

Individual SHAP values are local explanations.

A global summary can be obtained by aggregating the absolute SHAP magnitudes across many observations.

Here we use the mean absolute SHAP value for each transformed feature.


In [ ]:
mean_abs_shap = pd.Series(
    np.abs(
        survived_shap.values
    ).mean(axis=0),
    index=feature_names
).sort_values(
    ascending=False
)

global_shap_table = (
    mean_abs_shap
    .rename(
        "mean_absolute_SHAP"
    )
    .reset_index()
    .rename(
        columns={
            "index": "feature"
        }
    )
)

display(
    global_shap_table
)


In [ ]:
plot_table = (
    global_shap_table
    .sort_values(
        "mean_absolute_SHAP",
        ascending=True
    )
)

plt.figure(figsize=(9, 5))

plt.barh(
    plot_table["feature"],
    plot_table["mean_absolute_SHAP"]
)

plt.xlabel(
    "Mean absolute SHAP value"
)

plt.title(
    "Global SHAP summary for class 1: survived"
)

plt.show()


### Interpreting the global SHAP summary

A larger mean absolute SHAP value means that, across the explained test observations, that transformed feature tended to receive larger-magnitude attributions for the class-1 model output.

This is a summary of **attribution magnitude**.

It does not show causal influence, and it does not by itself show the direction of a feature's effect.


## 8. Local SHAP explanation

We now explain one test prediction.

A local SHAP explanation shows how the explainer decomposes the selected model output for one observation relative to the reference value.


In [ ]:
# Prefer a Random Forest misclassification if one exists.
misclassified_positions = np.flatnonzero(
    forest_test_pred != y_test.to_numpy()
)

if len(misclassified_positions) > 0:
    sample_position = int(
        misclassified_positions[0]
    )
    selection_reason = "misclassified Random Forest test case"
else:
    sample_position = 0
    selection_reason = "first test case"

raw_sample = X_test_raw.iloc[
    [sample_position]
]

predicted_class = int(
    forest_model.predict(
        X_test_df.iloc[
            [sample_position]
        ]
    )[0]
)

predicted_probability = (
    forest_model.predict_proba(
        X_test_df.iloc[
            [sample_position]
        ]
    )[0, 1]
)

true_class = int(
    y_test.iloc[
        sample_position
    ]
)

print(
    "Selected:",
    selection_reason
)

display(
    raw_sample
)

print(
    "Observed outcome:",
    "survived"
    if true_class == 1
    else "did not survive"
)

print(
    "Predicted class:",
    "survived"
    if predicted_class == 1
    else "did not survive"
)

print(
    "Predicted probability of survival:",
    round(
        predicted_probability,
        3
    )
)


In [ ]:
# Display a waterfall plot for the selected observation.
shap.plots.waterfall(
    survived_shap[
        sample_position
    ],
    max_display=10
)


### How to read the local explanation

The waterfall plot starts from the SHAP reference value and adds the feature attributions for the selected observation.

Depending on the model output represented by the explainer:

- positive SHAP values move the explained class-1 output upward relative to the reference;
- negative SHAP values move it downward.

The plot explains the Random Forest's output under the SHAP framework. It does not explain the real-world cause of survival.


## 9. Check the SHAP additivity relationship

A useful internal consistency check is the SHAP additivity relationship:

**reference value + sum of SHAP values ≈ explained model output**

The code below compares the reconstructed SHAP output with the Random Forest's predicted probability for class 1.


In [ ]:
# Reference value for this observation.
base_value = float(
    np.asarray(
        survived_shap.base_values[
            sample_position
        ]
    ).squeeze()
)

# Sum of feature attributions.
sum_shap = float(
    np.sum(
        survived_shap.values[
            sample_position
        ]
    )
)

reconstructed_output = (
    base_value + sum_shap
)

print(
    "Reference value:",
    round(
        base_value,
        6
    )
)

print(
    "Sum of SHAP values:",
    round(
        sum_shap,
        6
    )
)

print(
    "Reconstructed SHAP output:",
    round(
        reconstructed_output,
        6
    )
)

print(
    "Random Forest P(survived):",
    round(
        predicted_probability,
        6
    )
)


### Interpreting the additivity check

If the values match closely, the local SHAP decomposition is internally consistent with the explained class-1 model output.

This check concerns **fidelity to the model output being explained**.

It does not establish that the prediction is correct or that the attributions correspond to real-world causal effects.


## 10. Intrinsic vs. post-hoc explanation

We can now compare the two explanation settings directly.

| Aspect | Small decision tree | Random Forest + SHAP |
|---|---|---|
| Model | Single small tree | Ensemble of many trees |
| Source of explanation | Fitted model structure | Additional explainer applied after training |
| Local explanation | Exact tree path | SHAP attribution decomposition |
| Global view | Inspect complete tree | Aggregate local SHAP values |
| Approximation / framework dependence | Direct model inspection | Depends on SHAP formulation and explainer |
| Causal interpretation | No | No |

The distinction is not that one explanation is automatically better.

The distinction is **where the explanation comes from and what assumptions or machinery are required to produce it**.


## 11. Scientific cautions

### 1. SHAP explains the fitted model

A large SHAP attribution means that a feature received a large attribution for the explained model output under the selected explainer and setup.

It does not imply causal effect.

### 2. The reference value matters

SHAP values are defined relative to a reference or expected model output. Different explainer configurations or background choices can influence the explanation.

### 3. One local explanation is not the whole model

A waterfall plot explains one prediction. It should not be generalised automatically to all observations.

### 4. Global SHAP summaries are aggregates

Mean absolute SHAP values summarise attribution magnitude across observations. They do not show a causal ordering of variables.

### 5. Post-hoc explanation does not make the model intrinsically interpretable

Applying SHAP to a Random Forest provides information about model behaviour, but the underlying ensemble remains structurally complex.


## 12. Student task

Select one additional test observation.

For that observation:

1. report the observed outcome;
2. report the Random Forest prediction;
3. report the predicted probability of survival;
4. display the local SHAP waterfall plot;
5. identify the three largest absolute SHAP attributions;
6. state whether each attribution moves the explained class-1 output upward or downward;
7. verify the SHAP additivity relationship.

Then compare the local SHAP explanation with an exact decision path from the small decision tree for the same observation.

Answer:

- What information is available directly from the decision tree?
- What additional method is required for the Random Forest?
- Are the two explanations describing the same fitted model?
- Can either explanation be interpreted causally?


## 13. Questions for discussion

1. Why is the small tree considered intrinsically interpretable?
2. Why is SHAP a post-hoc explanation in this exercise?
3. Does applying SHAP make the Random Forest intrinsically interpretable?
4. What is the difference between an exact tree path and a SHAP attribution decomposition?
5. Why can local SHAP values be aggregated to form a global summary?
6. Why does a global SHAP summary still depend on the explained observations and explainer setup?
7. What does the SHAP additivity check demonstrate?
8. What does it **not** demonstrate?
9. Why should neither explanation be interpreted causally?


## 14. Takeaway

Intrinsic and post-hoc explainability provide different ways of investigating model behaviour.

In this exercise:

- the small decision tree can be inspected directly;
- its structure and local decision paths come from the fitted model itself;
- the Random Forest is harder to inspect as a complete ensemble;
- SHAP provides post-hoc feature attributions for the Random Forest;
- local SHAP values can be aggregated to summarise broader attribution patterns;
- the SHAP additivity relationship can be checked against the explained model output.

Post-hoc explanations provide information about a fitted model, but they do not make the underlying model intrinsically interpretable.

